# Module 4 · Deploying the Container to Vertex AI
### FastAPI for ML Engineers — Bootcamp Session (Part 4 of 4)

**Time budget:** ~60 minutes
**Prereq:** Module 3 (you should have a working Docker image). **You'll need a GCP project with billing enabled, the `gcloud` CLI installed and authenticated, and the Vertex AI + Artifact Registry APIs enabled** to actually run the commands in this notebook end-to-end. Everything is written so you can read and understand it either way — commands are annotated with what they do and what to expect.

## Agenda
| Time | Topic |
|---|---|
| 0:00–0:10 | Vertex AI serving model: what a "custom container" actually needs |
| 0:10–0:20 | Push the image to Artifact Registry |
| 0:20–0:35 | Register a `Model` in Vertex AI, create an `Endpoint`, deploy |
| 0:35–0:50 | Call the live endpoint (Python SDK + REST) |
| 0:50–0:55 | Monitoring, scaling, cost, cleanup |
| 0:55–1:00 | Exercise + wrap-up |

## Learning objectives
- Know exactly what contract Vertex AI expects from a custom-container model (routes, port, request/response shape)
- Push a Docker image to Artifact Registry
- Upload a `Model` and deploy it to an `Endpoint` in Vertex AI
- Send prediction requests to a live Vertex endpoint from Python and via REST
- Understand the cost/scaling knobs and how to tear everything down cleanly

## 4.1 What does Vertex AI actually expect from your container?

Vertex AI's **custom container** prediction route is deliberately close to what we already built — that's not an accident, it's the near-universal shape (SageMaker, KServe, and Vertex all converge on something very similar):

| Requirement | Why |
|---|---|
| Container listens on `$PORT` (Vertex sets this env var; default `8080`) | Vertex's routing layer forwards traffic to this port |
| A **health route** (default expected at `/health`, but configurable) returning `200` when ready | Used for readiness/liveness checks before routing traffic |
| A **predict route** (default expected at `/predict`, configurable) accepting `POST` with a JSON body shaped `{"instances": [...]}` | This is Vertex's standard prediction request envelope |
| Response shaped `{"predictions": [...]}` | Vertex's standard prediction response envelope |
| Image is in **Artifact Registry** (or Container Registry) in the same GCP project (or one Vertex can pull from) | Vertex pulls the image from there when deploying |

Compare this to our Module 2/3 `/predict/batch` endpoint — the shape is nearly identical (`instances` in, `predictions` out). That's exactly why we built it that way early: **you can either adapt Vertex's expected envelope to match your existing API, or adjust your API slightly to match Vertex's envelope exactly.** We'll do the latter here since it lets you use Vertex's Python SDK helper methods (`endpoint.predict(instances=...)`) with zero glue code.

## 4.2 Adjusting `main.py` to match Vertex's envelope exactly

Two small changes from Module 3's `main.py`:
1. Accept `{"instances": [{...}, {...}]}` at `/predict` (not a single object) — Vertex always sends a batch, even for one row.
2. Return `{"predictions": [...]}`.

This is a tiny diff — copy this over your Module 3 `app/main.py` before building the image you'll push in this module.

In [ ]:
import os
os.makedirs("/home/claude/fastapi_docker_project/app/model_artifacts", exist_ok=True)

In [ ]:
%%writefile /home/claude/fastapi_docker_project/app/main.py
import os
import joblib
import numpy as np
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List

MODEL_PATH = os.path.join(os.path.dirname(__file__), "model_artifacts", "model.joblib")
MODEL_VERSION = "fraud_v1_logreg"

ML_MODELS = {}

@asynccontextmanager
async def lifespan(app: FastAPI):
    ML_MODELS["fraud_model"] = joblib.load(MODEL_PATH)
    yield
    ML_MODELS.clear()

app = FastAPI(title="Fraud Model Serving API — Vertex-ready", version="1.0.0", lifespan=lifespan)


class Instance(BaseModel):
    feature_1: float = Field(..., description="Account age in years")
    feature_2: float = Field(..., description="Number of transactions in last 24h")
    feature_3: float = Field(..., description="Average transaction amount")


class VertexPredictRequest(BaseModel):
    instances: List[Instance]


class VertexPredictResponse(BaseModel):
    predictions: List[dict]


@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": "fraud_model" in ML_MODELS}


@app.post("/predict", response_model=VertexPredictResponse)
def predict(request: VertexPredictRequest):
    model = ML_MODELS.get("fraud_model")
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")

    if not request.instances:
        raise HTTPException(status_code=400, detail="instances must not be empty")

    features = np.array([[r.feature_1, r.feature_2, r.feature_3] for r in request.instances])

    try:
        preds = model.predict(features)
        probas = model.predict_proba(features)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference failed: {type(e).__name__}")

    predictions = [
        {"prediction": int(p), "probability": float(probas[i][p]), "model_version": MODEL_VERSION}
        for i, p in enumerate(preds)
    ]
    return VertexPredictResponse(predictions=predictions)

Rebuild the image after this change (from Module 3's directory):
```bash
cd fastapi_docker_project
docker build -t fraud-model-api:vertex .
```

## 4.3 Push the image to Artifact Registry

Set your project variables once, then create a repository (one-time setup) and push.

In [ ]:
%%bash
# --- one-time setup: fill these in for your project ---
export PROJECT_ID="your-gcp-project-id"
export REGION="us-central1"
export REPO_NAME="ml-serving-repo"

gcloud config set project "$PROJECT_ID"

# Enable the APIs we need (idempotent if already enabled)
gcloud services enable artifactregistry.googleapis.com aiplatform.googleapis.com

# Create an Artifact Registry Docker repository (one-time; skip if it already exists)
gcloud artifacts repositories create "$REPO_NAME" \
    --repository-format=docker \
    --location="$REGION" \
    --description="ML model serving images"

In [ ]:
%%bash
export PROJECT_ID="your-gcp-project-id"
export REGION="us-central1"
export REPO_NAME="ml-serving-repo"
export IMAGE_NAME="fraud-model-api"
export IMAGE_TAG="v1"

# Authenticate Docker against Artifact Registry (one-time per machine)
gcloud auth configure-docker "${REGION}-docker.pkg.dev" --quiet

export FULL_IMAGE_URI="${REGION}-docker.pkg.dev/${PROJECT_ID}/${REPO_NAME}/${IMAGE_NAME}:${IMAGE_TAG}"

cd /home/claude/fastapi_docker_project
docker build -t "$FULL_IMAGE_URI" .
docker push "$FULL_IMAGE_URI"

echo "Pushed: $FULL_IMAGE_URI"

## 4.4 Register the Model and deploy an Endpoint

Two distinct Vertex AI resources, and it's worth keeping them conceptually separate:
- **`Model`** — a registered reference to your container image + serving config (which routes, which port). Uploading a Model doesn't cost anything by itself or serve traffic.
- **`Endpoint`** — the thing that actually runs machines and serves traffic. You deploy one or more Models to an Endpoint (this is also how you'd do canary/A-B traffic splitting between model versions later).

We'll use the `google-cloud-aiplatform` Python SDK — it's what you'll actually use day to day, more so than raw `gcloud` commands, once you're scripting this as part of a pipeline.

In [ ]:
!pip install -q --break-system-packages google-cloud-aiplatform
print("done")

In [ ]:
from google.cloud import aiplatform

PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
REPO_NAME = "ml-serving-repo"
IMAGE_NAME = "fraud-model-api"
IMAGE_TAG = "v1"

IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME}:{IMAGE_TAG}"

aiplatform.init(project=PROJECT_ID, location=REGION)

model = aiplatform.Model.upload(
    display_name="fraud-model-fastapi",
    serving_container_image_uri=IMAGE_URI,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health",
    serving_container_ports=[8080],
)
print("Model resource name:", model.resource_name)

### Deploying to an endpoint

`deploy()` provisions actual compute and can take **5–15 minutes** the first time. This is also where you set your machine type and autoscaling bounds — the knobs that determine both latency headroom and your bill.

In [ ]:
endpoint = model.deploy(
    deployed_model_display_name="fraud-model-v1",
    machine_type="n1-standard-2",   # right-size this to your model's actual CPU/memory needs
    min_replica_count=1,             # >=1 means no cold starts, but you pay for it 24/7
    max_replica_count=2,             # autoscaling ceiling — protects against a runaway bill
    traffic_split={"0": 100},        # 100% of traffic to this newly deployed model
)
print("Endpoint resource name:", endpoint.resource_name)

> **`min_replica_count=0` isn't available for all machine types/frameworks** — scale-to-zero support depends on the serving configuration. If you need to minimize idle cost for a low-traffic demo endpoint, check current scale-to-zero support for your setup rather than assuming it — this is exactly the kind of platform detail that changes over time, so verify against the current Vertex AI docs before relying on it in production.

## 4.5 Calling the live endpoint

Two ways: the Python SDK (simplest if you're already in a notebook/pipeline), and raw REST (useful for calling from any other language/service, or for debugging with `curl`).

In [ ]:
# --- Option A: Python SDK ---
response = endpoint.predict(
    instances=[
        {"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0},
        {"feature_1": -1.0, "feature_2": 40, "feature_3": 2.0},
    ]
)
print(response.predictions)

In [ ]:
# --- Option B: raw REST (what a non-Python service, or curl, would do) ---
import subprocess, json

PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
ENDPOINT_ID = endpoint.name  # numeric ID, available after deploy()

url = f"https://{REGION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/{REGION}/endpoints/{ENDPOINT_ID}:predict"

payload = {
    "instances": [
        {"feature_1": 2.0, "feature_2": 5, "feature_3": 10.0},
    ]
}

# Equivalent curl command, shown here rather than executed, since it needs a live gcloud
# identity token:
curl_cmd = f'''
curl -X POST \\
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \\
  -H "Content-Type: application/json" \\
  -d '{json.dumps(payload)}' \\
  "{url}"
'''
print(curl_cmd)

## 4.6 Monitoring, scaling, and cost

- **Cloud Logging** — every request/response and container stdout/stderr line is captured automatically; this is exactly why we logged to stdout back in Module 3 rather than to a file.
- **Cloud Monitoring** — Vertex AI Endpoints emit built-in metrics (prediction count, latency, error rate, replica count) with no extra instrumentation needed; you can wire alerts off these directly.
- **Autoscaling** — driven by `min_replica_count`/`max_replica_count` and CPU/traffic-based triggers. More replicas = more concurrent capacity, but also more cost — `min_replica_count=1` means you're paying for at least one machine every hour, even at zero traffic.
- **Cost drivers to know:** machine type (CPU/GPU/memory), number of replicas, and how long they run. A model with `min_replica_count=0` (where supported) costs nothing while idle but pays a cold-start latency penalty on the first request after idling.
- **Versioning/rollout:** deploy a new `Model` version to the same `Endpoint` with a partial `traffic_split` (e.g. `{"old_model_id": 90, "new_model_id": 10}`) to do a canary rollout before shifting 100%.

## 4.7 Cleanup — don't leave a billing surprise running

Undeploying and deleting are separate steps, and both matter: an undeployed Endpoint with no traffic still exists as a resource (free), but a *deployed* Endpoint keeps its replicas running and billing until you undeploy.

In [ ]:
# Undeploy the model from the endpoint (stops the running replicas / billing)
endpoint.undeploy_all()

# Delete the endpoint resource itself
endpoint.delete()

# Delete the model resource from the Model Registry
model.delete()

print("cleaned up")

In [ ]:
%%bash
# Optional: also remove the pushed image from Artifact Registry if you no longer need it
export PROJECT_ID="your-gcp-project-id"
export REGION="us-central1"
export REPO_NAME="ml-serving-repo"
export IMAGE_NAME="fraud-model-api"
export IMAGE_TAG="v1"

gcloud artifacts docker images delete \
    "${REGION}-docker.pkg.dev/${PROJECT_ID}/${REPO_NAME}/${IMAGE_NAME}:${IMAGE_TAG}" \
    --quiet

## Exercise 4 (10–15 min)

1. Deploy a **second** version of the model (retrain with a different `random_state` or hyperparameter, tag the image `:v2`, upload as a new `Model`), then deploy it to the **same endpoint** with a `traffic_split` of `{"v1_id": 80, "v2_id": 20}` to simulate a canary rollout.
2. Write a small Python loop that calls `endpoint.predict()` 20 times and tallies which underlying model version answered each time (hint: Vertex doesn't tell you directly per-response which deployed model served it under a traffic split — think about how you'd instrument this yourself, e.g. by having each container version return its own `model_version` string in the response body, which we already do).
3. Bonus: set `min_replica_count=0` (if supported for your chosen machine type) and measure the cold-start latency of the first request after a few idle minutes vs. a warm request.

## Recap — the whole pipeline, end to end

1. **Module 1:** built a FastAPI app, understood HTTP semantics, protected it with API key / Basic / OAuth2+JWT auth.
2. **Module 2:** built a real `/predict` endpoint around a trained model, with Pydantic validation, batch support, and correct error handling.
3. **Module 3:** packaged that exact service into a Docker image — reproducible, portable, secrets handled at runtime, listening on `$PORT`.
4. **Module 4:** pushed that image to Artifact Registry and deployed it as a scalable, monitored Vertex AI Endpoint, callable from anywhere.

This is the same path a production model takes at most companies doing MLOps seriously — the only things that change project to project are the model itself and which managed platform sits at the end (Vertex AI here; SageMaker, KServe/GKE, or Azure ML follow the same shape with different resource names).